# Follow-ups — the three things that can't run locally

Each part is independent; run whichever you need and paste the output back.

| part | what | why it's here |
| --- | --- | --- |
| **A** | Re-run Phase 4 (Kmult) after the stripe recalibration | needs R + `geomorph` |
| **B** | *patternize* equivalence check | needs R + the `patternize` package |
| **C** | Build the GBIF derived-dataset table | ~1,878 rate-limited API calls |

**Use a CPU runtime.** No GPU needed for any part.

## Setup (run first, for any part)

> **If you hit `Transport endpoint is not connected`** — that is the Google Drive
> FUSE mount collapsing, not a git problem, and it can corrupt the repo on Drive if
> it happens mid-write. It cannot be fixed from inside the same session:
>
> 1. **Runtime → Restart session**
> 2. Re-run the mount cell
> 3. Run the **Recovery** cell below, which verifies the repo and re-clones it if
>    it is damaged
>
> Nothing important lives only on Drive — every result file is in the repo. The one
> exception is `data/extracted_fish/` (~480 MB, gitignored), which the recovery cell
> preserves. **Part A does not need it at all**, so a Kmult re-run can proceed even
> if those images are lost.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")


def git(*args, check=True):
    """Runs git and always shows its real output - a bare check=True hides it."""
    r = subprocess.run(["git", "-C", str(PROJECT_DIR), *args],
                       capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}) - see above")
    return r


if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    # This clone predates outputs/ and reports/ being tracked in git, so it
    # holds untracked local copies of files that now exist in the repo, and a
    # plain pull aborts on them. They are all either identical to the tracked
    # versions or stale (the repo was re-run after they were made), so the fix
    # is to move them aside - to a timestamped backup folder, not /dev/null -
    # and pull again.
    import re, shutil, datetime

    result = git("pull", check=False)
    if result.returncode:
        blocked = re.findall(r"^\t(.+)$", result.stderr, flags=re.M)
        if not blocked:
            raise RuntimeError("pull failed for a reason other than blocking files - see above")
        stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        backup = PROJECT_DIR.parent / f"surgeonfish-prepull-backup-{stamp}"
        print(f"\n{len(blocked)} untracked file(s) blocking the pull.")
        print(f"Moving them to {backup} (nothing deleted), then retrying...\n")
        for rel in blocked:
            src = PROJECT_DIR / rel
            if not src.exists():
                continue
            dest = backup / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dest))
        git("pull")

git("log", "--oneline", "-1")

### Recovery — run this if the mount dropped or the pull left the repo broken

Checks the repo is readable and consistent (`git fsck`). If it isn't, moves
`data/extracted_fish/` aside, re-clones from scratch, and puts the images back.

In [ ]:
import shutil, subprocess
from pathlib import Path

if not Path("/content/drive/MyDrive").is_dir():
    raise SystemExit(
        "Drive is not mounted. Runtime -> Restart session, then re-run the mount cell."
    )

def repo_is_healthy(path):
    if not (path / ".git").is_dir():
        return False
    probe = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"],
                           capture_output=True, text=True)
    if probe.returncode:
        return False
    fsck = subprocess.run(["git", "-C", str(path), "fsck", "--connectivity-only"],
                          capture_output=True, text=True)
    return fsck.returncode == 0

if repo_is_healthy(PROJECT_DIR):
    print("Repo looks healthy - no recovery needed.")
else:
    print("Repo is missing or damaged. Re-cloning...\n")
    images = PROJECT_DIR / "data/extracted_fish"
    parked = PROJECT_DIR.parent / "extracted_fish_rescued"
    if images.is_dir() and not parked.exists():
        print("Preserving data/extracted_fish (this takes a minute)...")
        shutil.move(str(images), str(parked))
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    if parked.is_dir():
        (PROJECT_DIR / "data").mkdir(parents=True, exist_ok=True)
        shutil.move(str(parked), str(PROJECT_DIR / "data/extracted_fish"))
        print("Restored data/extracted_fish")
    print("\nRe-clone complete.")

subprocess.run(["git", "-C", str(PROJECT_DIR), "log", "--oneline", "-1"])

In [ ]:
%cd {PROJECT_DIR}

In [ ]:
%pip install -q "biopython>=1.81"
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("ready")

---
# Part A — Re-run Phase 4 after the stripe recalibration

`StripeConfig` was recalibrated against the **real SAM 2 masks** (recall 14.3% → 50.0%
at identical precision, validated held-out over 200 split-half trials). Phases 2 and 3
were already re-run locally and their outputs are in the repo you just pulled — so this
only needs to redo the part that requires R.

**What to watch for:** stripe's phylogenetic-signal result was previously *null and
uninterpretable* — the detector was missing ~86% of real stripes, so a null couldn't be
distinguished from "not measured well enough." With recall now at 50%, whatever comes
back is a far more meaningful test. A still-null stripe result would become a real
biological finding rather than an instrument artifact.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", force=True)

from phylo_comparison.config import ExportConfig, FeaturePrepConfig
from phylo_comparison.pipeline import run as run_prep

species_order = run_prep(
    FeaturePrepConfig(species_features_csv_path=PROJECT_DIR / "reports/species_features.csv"),
    ExportConfig(
        output_dir=PROJECT_DIR / "outputs/phase4",
        tree_path=PROJECT_DIR / "data/phylogeny/actinopt_12k_treePL.tre",
        species_coverage_csv_path=PROJECT_DIR / "data/phylogeny/species_coverage.csv",
    ),
)
print(f"prepared {len(species_order)} species")

In [ ]:
# Same for the 47-species sensitivity set, so the R script runs both.
# Phases 2 and 3 were already re-run locally and their outputs are in the
# repo, so this only rebuilds the Kmult-ready matrices - no need for the
# mask PNGs that Phase 3 aggregation would otherwise require.
run_prep(
    FeaturePrepConfig(
        species_features_csv_path=PROJECT_DIR / "reports/species_features_min5.csv"
    ),
    ExportConfig(
        output_dir=PROJECT_DIR / "outputs/phase4_min5",
        tree_path=PROJECT_DIR / "data/phylogeny/actinopt_12k_treePL.tre",
        species_coverage_csv_path=PROJECT_DIR / "data/phylogeny/species_coverage.csv",
    ),
)
print("sensitivity set prepared")

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
# geomorph compiles from source on first install - 5-15 minutes. Be patient.
if (!requireNamespace("geomorph", quietly = TRUE)) {
  install.packages("geomorph", dependencies = TRUE, Ncpus = 2)
}
library(geomorph)
packageVersion("geomorph")

In [ ]:
project_dir_str = str(PROJECT_DIR)

In [ ]:
%%R -i project_dir_str
setwd(project_dir_str)
source("r/phase4_kmult.R")

In [ ]:
import pandas as pd
p = PROJECT_DIR / "outputs/phase4"
print("Kmult (primary):");        display(pd.read_csv(p / "kmult_results.csv"))
print("Mantel (secondary):");     display(pd.read_csv(p / "mantel_results.csv"))
print("Sensitivity 49 vs 47:");   display(pd.read_csv(p / "sensitivity_comparison.csv"))

---
# Part B — *patternize* equivalence check

This closes the project's **largest unquantified risk**: `pattern_extractor`'s colour
clustering is a Python reimplementation of *patternize*'s reference-initialised k-means
(Van Belleghem et al. 2018) that has never been checked against the original R package —
and colour is the dimension carrying the only significant result.

The relevant patternize function, confirmed against the CRAN manual (not guessed):

```r
kImage(image, k = 5, startCenter = NULL, maskToNA = NULL, kmeansOnAll = FALSE)
```

`startCenter` — "a matrix of cluster centres to start k-means clustering from" — is
exactly the reference-initialisation mechanism this project reimplemented.

**What is and isn't being tested.** This project deliberately diverged from patternize in
v2.2.1 by clustering in a lightness-invariant hue/saturation space rather than raw RGB —
a documented design change, not a porting error. So the check runs **both
implementations on RGB**, isolating the question that actually matters: *is our
reference-initialised k-means machinery equivalent to patternize's?* A disagreement here
would be a real porting bug; the hue/sat difference is out of scope by design.

In [ ]:
# patternize -> ClusterR -> gmp, and the gmp R package is a binding to the GNU
# MP *C library*, which Colab does not ship. Without this apt step the R install
# fails with "installation of 3 packages failed: gmp, ClusterR, patternize"
# while every other dependency compiles fine.
!apt-get -qq install -y libgmp-dev libmpfr-dev > /dev/null 2>&1
print("system libraries installed")

In [ ]:
%%R
# patternize pulls in raster/sp/Morpho/ClusterR - allow 10-20 minutes.
for (pkg in c("gmp", "ClusterR", "patternize")) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE, Ncpus = 2)
  }
}

# Report what actually installed rather than dying on library() with a bare
# "no package called patternize" - the useful information is which link in the
# dependency chain broke.
for (pkg in c("gmp", "ClusterR", "patternize")) {
  ok <- requireNamespace(pkg, quietly = TRUE)
  cat(sprintf("%-12s %s\n", pkg, if (ok) "OK" else "FAILED"))
}

if (requireNamespace("patternize", quietly = TRUE)) {
  library(patternize)
  cat("\npatternize", as.character(packageVersion("patternize")), "ready\n")
} else {
  cat("\npatternize did not install. Part B cannot run; Parts A and C are unaffected.\n")
}

In [ ]:
# The comparison inputs are committed under outputs/patternize_check/ and were
# generated locally by scratchpad/make_patternize_inputs.py. Each PNG holds ONLY
# the masked-in pixels of one crop, reshaped into a rectangle - so there is no
# background for kImage to include and ours to exclude, which is what made the
# first version of this check uninterpretable.
import pandas as pd

ref = pd.read_csv(PROJECT_DIR / "outputs/patternize_check/python_reference.csv")
display(ref)
print("Both implementations will cluster these exact pixel sets.")

In [ ]:
%%R -i project_dir_str
setwd(project_dir_str)
library(patternize); library(raster)

work <- "outputs/patternize_check"
pngs <- list.files(work, pattern = "\.png$", full.names = TRUE)

cat(sprintf("Comparing %d image(s)

", length(pngs)))
worst <- 0

for (png in pngs) {
  name <- sub("\.png$", "", basename(png))
  centres <- as.matrix(read.csv(file.path(work, paste0(name, "_centres.csv")),
                                header = FALSE))
  py <- as.numeric(read.csv(file.path(work, paste0(name, "_fractions.csv")),
                            header = FALSE)[, 1])

  img <- raster::stack(png)

  # Our centres are in 0-255. Some raster readers return 0-1, which would make
  # startCenter meaningless - so check rather than assume, and rescale if needed.
  rng <- range(raster::values(img), na.rm = TRUE)
  start <- centres
  if (rng[2] <= 1.5) {
    start <- centres / 255
    cat(sprintf("%s: raster is 0-1 (range %.3f-%.3f); rescaled centres
",
                name, rng[1], rng[2]))
  }

  res <- kImage(img, k = nrow(start), startCenter = start)

  km <- NULL
  for (part in res) if (inherits(part, "kmeans")) km <- part
  if (is.null(km)) {
    cat("No kmeans component found. Actual structure:
")
    str(res, max.level = 2)
    next
  }
  r_frac <- sort(km$size / sum(km$size), decreasing = TRUE)
  diff <- max(abs(r_frac - py))
  worst <- max(worst, diff)

  cat(sprintf("
--- %s ---
", name))
  cat("patternize kImage :", sprintf("%.4f", r_frac), "
")
  cat("pattern_extractor :", sprintf("%.4f", py), "
")
  cat("max abs difference:", sprintf("%.4f", diff), "
")
}

cat(sprintf("

Worst disagreement across all images: %.4f
", worst))
cat("
Interpretation: both implementations clustered the SAME pixels from the
")
cat("SAME starting centres, so this number is a genuine algorithmic comparison.
")
cat("k-means is iterative, so small differences are expected from convergence
")
cat("details; a large disagreement (say >0.05) would indicate a real porting bug.
")

---
# Part C — Build the GBIF derived-dataset table

GBIF's guidance asks for a registered **derived dataset** when occurrences are pulled via
the search API rather than a bulk download. This is the project's only hard publication
blocker.

`reports/image_sourcing_log.csv` records an `occurrence_key` per image but not the
`datasetKey` that registration needs, so each occurrence has to be resolved against the
GBIF API. **~1,878 requests at 1/second ≈ 30 minutes** — the rate limit matches this
project's own stated GBIF policy (see CLAUDE.md). The cell saves partial progress and
can be re-run to resume.

**This cell only produces the table.** Registering it needs your GBIF account:
https://www.gbif.org/derived-dataset/register

In [ ]:
import csv, json, time
import urllib.request
from collections import Counter
from pathlib import Path

SRC = PROJECT_DIR / "reports/image_sourcing_log.csv"
CACHE = PROJECT_DIR / "outputs/gbif_occurrence_to_dataset.json"
OUT = PROJECT_DIR / "outputs/gbif_derived_dataset.csv"
UA = "Surgeonfish-Phenomics/1.0 (research; rgupta25@charlotte.edu)"

keys = []
with open(SRC, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        k = (row.get("occurrence_key") or "").strip()
        if k:
            keys.append(k)
unique = sorted(set(keys))
print(f"{len(keys)} logged images, {len(unique)} unique occurrence keys")

cache = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}
todo = [k for k in unique if k not in cache]
print(f"{len(cache)} already resolved, {len(todo)} to fetch "
      f"(~{len(todo)/60:.0f} min at 1 req/sec)")

for i, key in enumerate(todo, 1):
    try:
        req = urllib.request.Request(
            f"https://api.gbif.org/v1/occurrence/{key}", headers={"User-Agent": UA}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            cache[key] = json.load(resp).get("datasetKey")
    except Exception as exc:
        cache[key] = None
        print(f"  {key}: {exc}")
    time.sleep(1.0)   # matches this project's stated GBIF rate limit
    if i % 50 == 0:
        CACHE.write_text(json.dumps(cache), encoding="utf-8")
        print(f"  {i}/{len(todo)} resolved...", flush=True)

CACHE.write_text(json.dumps(cache), encoding="utf-8")

counts = Counter(cache[k] for k in unique if cache.get(k))
unresolved = sum(1 for k in unique if not cache.get(k))
with open(OUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["datasetKey", "occurrenceCount"])
    for dk, n in counts.most_common():
        w.writerow([dk, n])

print(f"\n{len(counts)} distinct source datasets, {sum(counts.values())} occurrences")
print(f"unresolved: {unresolved}")
print(f"wrote {OUT}")
print("\nRegister at https://www.gbif.org/derived-dataset/register using that CSV.")

---
## When you're done

Paste back whichever outputs you ran — especially **Part A's three tables** (the stripe
recalibration is the one that could change a scientific conclusion) and **Part B's
fraction comparison**. Commit anything new under `outputs/` so the tracked results stay
in sync with the README.